### Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parents[1]))

from utils.loader import get_loader
from utils.notebook import FilenameLoader

### Run Clean
Define dataset, eps, and the sub-epoch/epoch/vehicle params

In [ ]:
clean_ckpt_file, data_file, save_name, ckpt_name = FilenameLoader.const_pos()
pgd_eps = 0.04
training_params = "30-30-200"

ckpt_name, pgd_eps, training_params

#### Original

In [ ]:
# Run clean test on cleanly trained (no adv training) model
save_dir = f"defenses/fed/data-test/{save_name}-clean-2"
loader_clean = get_loader(ckpt_file="ConstantPos-final-2.ckpt",
                          data_file=data_file,
                          save_dir=save_dir)

out = loader_clean.test_clean(filename="clean")

#### General / Targeted

In [ ]:
# Run clean test on general model
ckpt_file= f"adv_trained/{ckpt_name}-General-{pgd_eps}-{training_params}.ckpt"
save_dir = f"defenses/fed/data-test/{save_name}-general-{pgd_eps}-{training_params}"

loader_clean = get_loader(ckpt_file=ckpt_file,
                          data_file=data_file,
                          save_dir=save_dir)
out = loader_clean.test_clean(filename="clean")

In [ ]:
# Run clean test on targeted model
ckpt_file= f"adv_trained/{ckpt_name}-Targeted-{pgd_eps}-{training_params}.ckpt"
save_dir = f"defenses/fed/data-test/{save_name}-targeted-{pgd_eps}-{training_params}"

loader_clean = get_loader(ckpt_file=ckpt_file,
                          data_file=data_file,
                          save_dir=save_dir)
out = loader_clean.test_clean(filename="clean")

### Create Table

In [ ]:
import json
import pandas as pd

_, _, save_name, _ = FilenameLoader.const_pos()
training_params = "30-30-200"
pgd_eps = 0.05
sec_pgd_eps=0.01
folders = [f"{save_name}-clean-2", 
           f"{save_name}-general-{pgd_eps}-{training_params}", 
           f"{save_name}-targeted-{pgd_eps}-{training_params}",
           f"{save_name}-general-{sec_pgd_eps}-{training_params}",
           f"{save_name}-targeted-{sec_pgd_eps}-{training_params}"]

# folders = ["randpos-clean", "randpos-general-5-5-5", "randpos-targeted-5-5-5"]
root = pathlib.Path("./data-test")
metric = ["f1", "precision", "recall", "accuracy", "falseNegativeRate"]

rows = []
for folder in folders:
    with open(root / folder / "clean.json", "r") as file:
        out = json.load(file)["wrapper"]
    rows.append({m: out[m] for m in metric})

df = pd.DataFrame(rows, index=folders)
df